# Read_File Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Create, then read it all.** `write_text` puts the file on disk; `.read()` slurps the whole thing back as one string.

In [ ]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

# Step 1: create the practice file (borrowed from Lesson 2).
text = "Roses are red\nViolets are blue\nPython reads files for you\n"
path = Path("sample_data", "poem.txt")
path.write_text(text, encoding="utf-8")

# Step 2: read the whole thing back.
with open(path, encoding="utf-8") as f:
    content = f.read()
print(content)
print("Characters in file:", len(content))   # 58

**2. Line by line by hand.** Each `.readline()` returns the next line, `\n` included — display helpers like `.strip()` tidy it for humans.

In [ ]:
with open("sample_data/poem.txt", encoding="utf-8") as f:
    line1 = f.readline()
    line2 = f.readline()
    line3 = f.readline()

print(repr(line1))        # 'Roses are red\n'  <- the \n rides along
print(line2.strip())      # Violets are blue
print(line3.upper())      # PYTHON READS FILES FOR YOU

**3. All lines at once.** `.readlines()` returns a list of line strings; `enumerate(..., start=1)` supplies the numbering.

In [ ]:
with open("sample_data/poem.txt", encoding="utf-8") as f:
    lines = f.readlines()

print("Lines in file:", len(lines))          # 3

for number, line in enumerate(lines, start=1):
    print(number, "->", line.strip())
# 1 -> Roses are red
# 2 -> Violets are blue
# 3 -> Python reads files for you

## Part 2 — Practice

**4. Character budget.** Both reads share one cursor: `.read(11)` consumes eleven characters, and the second `.read()` continues from wherever it stopped.

In [ ]:
with open("sample_data/poem.txt", encoding="utf-8") as f:
    head = f.read(11)     # first 11 characters...
    rest = f.read()       # ...then everything AFTER them

print(repr(head))                # 'Roses are r'
print(len(rest), "chars left")   # 47 chars left

# Both reads advance the SAME cursor - the file object remembers
# how far it has consumed.

**5. The missing file.** `FileNotFoundError` is caught and handled politely; `Path.exists()` answers the same question without raising.

In [ ]:
from pathlib import Path

try:
    with open("sample_data/unicorn.txt", encoding="utf-8") as f:
        print(f.read())
except FileNotFoundError:
    print("No unicorn file - carrying on politely.")

guess = Path("sample_data", "unicorn.txt")
print(guess.exists())    # False

**6. Close encounters.** Manual closing works but relies on your memory; `with` closes the file on every exit path, exceptions included.

In [ ]:
# Manual close: works, but YOU must remember it.
f = open("sample_data/poem.txt", encoding="utf-8")
text = f.read()
print(f.closed)    # False - still open
f.close()
print(f.closed)    # True - closed by hand

# with closes automatically, even if an error fires mid-block.
with open("sample_data/poem.txt", encoding="utf-8") as g:
    g.read()
print(g.closed)    # True - closed for us on exit

**7. One-line pathlib read.** `read_text()` opens, reads and closes in one expression; `split()` counts words and `splitlines()[-1]` grabs the last line.

In [ ]:
from pathlib import Path

text = Path("sample_data", "poem.txt").read_text(encoding="utf-8")

print(len(text.split()))          # 11
print(text.splitlines()[-1])      # Python reads files for you

## Part 3 — Challenge

**8. Stream the sensor log.** Streaming with `for line in f:` holds one line in memory at a time while tracking the maximum temperature.

In [ ]:
from pathlib import Path

readings = ["08:00 21.4", "10:00 24.0", "12:00 27.3",
            "14:00 29.9", "16:00 25.6"]

with open("sample_data/sensor.log", "w", encoding="utf-8") as f:
    for reading in readings:
        f.write(reading + "\n")           # setup borrows "w" mode

hottest = None                             # highest temperature seen so far
count = 0
with open("sample_data/sensor.log", encoding="utf-8") as f:
    for line in f:                         # ONE line alive per iteration
        count += 1
        temp = float(line.split()[1])
        if hottest is None or temp > hottest:
            hottest = temp

print(count, "readings")                   # 5 readings
print("Hottest:", hottest)                 # Hottest: 29.9

# Streaming keeps memory flat: a 50 GB log costs one line's worth of RAM,
# whereas .read()/.readlines() would try to fit it all in memory.

**9. Encoding round-trip.** Machine-default encodings differ per computer; passing `encoding="utf-8"` explicitly makes the file portable everywhere.

In [ ]:
from pathlib import Path

note = "Cafe bill: 250 taka ✓"
path = Path("sample_data", "note_unicode.txt")

# 1) Naive: machine-default encoding - may fail or garble elsewhere.
try:
    path.write_text(note)                  # no encoding given!
    print("Default encoding coped on THIS machine.")
except UnicodeEncodeError:
    print("Default encoding refused these characters!")

# 2) Professional: explicit UTF-8 works on every OS and machine.
path.write_text(note, encoding="utf-8")
print(path.read_text(encoding="utf-8"))    # Cafe bill: 250 taka ✓

# open() without encoding= falls back to local defaults (often cp1252
# on Windows), so a file fine on your laptop can crash elsewhere.